In [2]:
# Rebrand Automation: Fokus Janten → Arah Berita
# This notebook automates a repository-wide rebrand of a static news portal from Fokus Janten to Arah Berita.

# 1. Import Libraries and Define Paths
from pathlib import Path
import re
import json
import shutil

root = Path(".").resolve()
html_files = list(root.rglob("*.html"))
css_files = list(root.rglob("*.css"))
json_files = list(root.rglob("*.json"))
md_files = list(root.rglob("*.md"))

def read_text(path):
    return path.read_text(encoding="utf-8", errors="replace")

def write_text(path, content):
    path.write_text(content, encoding="utf-8")

changed_files = set()

# 2. Scan HTML and CSS Files
main_pages = [p for p in html_files if p.parent == root]
article_pages = [p for p in html_files if p.parent.name == "article"]
package_pages = [root / "package.json"]
if (root / "tools" / "package.json").exists():
    package_pages.append(root / "tools" / "package.json")
docs_pages = [root / "AUTOMATION_README.md", root / "GOOGLE_DRIVE_GUIDE.md", root / "GOOGLE_DRIVE_IMAGES_GUIDE.md", root / "netlify.toml"]

print(f"Main pages: {len(main_pages)}")
print(f"Article pages: {len(article_pages)}")
print(f"CSS files: {len(css_files)}")
print(f"Package files: {len(package_pages)}")
print(f"Docs files: {len([p for p in docs_pages if p.exists()])}")

# 3. Update Branding Strings
branding_replacements = [
    (r"Fokus\s?Janten", "Arah Berita"),
    (r"Fokusjanten", "ArahBerita"),
    (r"fokusjanten", "ArahBerita"),
    (r"FokusJanten", "ArahBerita"),
    (r"fokus janten", "arah berita"),
    (r"fokusjanten@gmail\.com", "ArahBerita@gmail.com"),
    (r"Fokusjanten@gmail\.com", "ArahBerita@gmail.com"),
]

social_replacements = [
    (r"facebook\.com/[^\s'\"]+", "facebook.com/arahberita"),
    (r"twitter\.com/[^\s'\"]+", "twitter.com/arahberita"),
    (r"instagram\.com/[^\s'\"]+", "instagram.com/arahberita"),
    (r"youtube\.com/[^\s'\"]+", "youtube.com/arahberita"),
]

for path in main_pages + article_pages + docs_pages + css_files + package_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in branding_replacements + social_replacements:
        try:
            content = re.sub(old, new, content)
        except re.error:
            content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 4. Apply Global Theme Colors
color_map = {
    "#0EA5E9": "#166534",
    "#0ea5e9": "#166534",
    "#22C55E": "#2D5016",
    "#22c55e": "#2D5016",
    "#0F172A": "#052E16",
    "#0f172a": "#052E16",
    "#FFC107": "#166534",
    "#ffc107": "#166534",
    "#FFCC00": "#166534",
    "#ffcc00": "#166534",
    "#FC0": "#166534",
    "#fc0": "#166534",
    "#1E2024": "#052E16",
    "#1e2024": "#052E16",
}

for path in css_files + main_pages + article_pages + docs_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in color_map.items():
        content = content.replace(old, new)
    if path.name == "style.css":
        content = content.replace("--primary: #0EA5E9;", "--primary: #166534;")
        content = content.replace("--secondary: #22C55E;", "--secondary: #2D5016;")
        content = content.replace("--dark: #0F172A;", "--dark: #052E16;")
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 5. Replace Logo References with Text Logo
text_logo = '<span style="font-weight: bold; color: #166534; font-size: 24px; letter-spacing: -0.5px;">ARAH<span style="color: #2D5016; font-weight: normal; font-size: 18px; margin-left: 2px;">BERITA</span></span>'

for path in main_pages + article_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    content = content.replace('src="img/logo.png"', '')
    content = content.replace('src="../img/logo.png"', '')
    content = content.replace('src="./img/logo.png"', '')
    content = content.replace('alt="logo"', 'alt="ArahBerita"')
    content = content.replace('alt="Logo"', 'alt="ArahBerita"')
    content = content.replace('alt="LOGO"', 'alt="ArahBerita"')
    content = content.replace('FOKUS<span style="color: #22C55E; font-weight: normal; font-size: 18px; margin-left: 2px;">JANTEN</span>', text_logo)
    content = content.replace('FOKUS<span style="color: #22C55E; font-weight: normal; font-size: 18px;">JANTEN</span>', text_logo)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 6. Fix Quotation and Encoding
encoding_replacements = {
    '“': '"',
    '”': '"',
    '‘': "'",
    '’': "'",
    '–': '-',
    '—': '-',
    '\uFFFD': ' ',
    '\u00A0': ' ',
}

for path in html_files + md_files + css_files + json_files:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in encoding_replacements.items():
        content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 7. Patch Article Pages
for path in article_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    content = content.replace('src="../img/logo.png"', '')
    content = content.replace('src="img/logo.png"', '')
    content = content.replace('src="./img/logo.png"', '')
    content = content.replace('alt="logo"', 'alt="ArahBerita"')
    content = content.replace('alt="Logo"', 'alt="ArahBerita"')
    if 'FOKUS<span style="color: #22C55E; font-weight: normal' in content:
        content = content.replace('FOKUS<span style="color: #22C55E; font-weight: normal; font-size: 18px; margin-left: 2px;">JANTEN</span>', text_logo)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 8. Update package.json Metadata
pkg_updates = {
    'package.json': {
        '"name": "fokusjanten"': '"name": "arahberita"',
        '"description": "Fokus Janten website"': '"description": "Arah Berita website"',
    },
    'tools/package.json': {
        '"name": "arahberita-article-generator"': '"name": "arahberita-article-generator"',
        '"description": "Generator artikel otomatis dari Google Sheets untuk Fokus Janten"': '"description": "Generator artikel otomatis dari Google Sheets untuk Arah Berita"',
        '"author": "Fokus Janten Team"': '"author": "Arah Berita Team"',
    }
}

for rel, updates in pkg_updates.items():
    path = root / rel
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in updates.items():
        content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 9. Update Documentation Files
doc_replacements = [
    ('Fokus Janten', 'Arah Berita'),
    ('fokusjanten', 'arahberita'),
    ('Fokusjanten', 'ArahBerita'),
    ('#FFCC00', '#166534'),
    ('#ffc107', '#166534'),
    ('#1E2024', '#052E16'),
    ('/fokusjanten/', '/arahberita/'),
]

for path in [p for p in docs_pages if p.exists()]:
    content = read_text(path)
    original = content
    for old, new in doc_replacements:
        content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

# 10. Verify Rebranding and Generate Counts
remaining_patterns = [
    'Fokus Janten',
    'Fokusjanten',
    'FokusJanten',
    'logo.png',
]
new_colors = ['#166534', '#052E16', '#2D5016']

remaining = {}
for pattern in remaining_patterns:
    remaining[pattern] = []
    for path in html_files + css_files + md_files + json_files:
        if not path.exists():
            continue
        content = read_text(path)
        if pattern in content:
            remaining[pattern].append(str(path))

color_presence = {c: 0 for c in new_colors}
for path in html_files + css_files:
    if not path.exists():
        continue
    content = read_text(path)
    for c in new_colors:
        if c in content:
            color_presence[c] += 1

counts = {
    'main_pages': len(main_pages),
    'article_pages': len(article_pages),
    'css_files': len(css_files),
    'package_files': len(package_pages),
    'docs_files': len([p for p in docs_pages if p.exists()]),
    'changed_files': len(changed_files),
}

print(json.dumps({
    'counts': counts,
    'remaining': {k: len(v) for k, v in remaining.items()},
    'new_color_presence': color_presence,
}, indent=2))
print('\nRebrand Arah Berita selesai ✅')

# 11. PowerShell Execution Script
shell_commands = [
    'Copy-Item -Path "articles.json" -Destination "articles.json.bak" -Force',
    'Get-ChildItem -Recurse -Include *.html,*.css,*.md,*.json | ForEach-Object {',
    '  $content = Get-Content $_.FullName -Raw -Encoding UTF8',
    '  $content = $content -replace "Fokus\\s?Janten", "Arah Berita"',
    '  $content = $content -replace "Fokusjanten", "ArahBerita"',
    '  $content = $content -replace "fokusjanten@gmail.com", "ArahBerita@gmail.com"',
    '  $content = $content -replace "#0EA5E9", "#166534"',
    '  $content = $content -replace "#22C55E", "#2D5016"',
    '  $content = $content -replace "#0F172A", "#052E16"',
    '  $content = $content -replace "#FFCC00", "#166534"',
    '  $content = $content -replace "#ffc107", "#166534"',
    '  $content = $content -replace "#1E2024", "#052E16"',
    '  $content = $content -replace "[“”]", "\""',
    '  $content = $content -replace "[‘’]", "\""',
    '  $content = $content -replace "[–—]", "-"',
    '  $content = $content -replace "\u00A0", " "',
    '  $content = $content -replace "\uFFFD", " "',
    '  $content = $content -replace "src=\"../img/logo.png\"", ""',
    '  $content = $content -replace "src=\"img/logo.png\"", ""',
    '  $content = $content -replace "alt=\"logo\"", "alt=\"ArahBerita\""',
    '  Set-Content -Path $_.FullName -Value $content -Encoding UTF8',
    '}',
]

print('\nPowerShell script to run:')
print('\n'.join(shell_commands))


Main pages: 9
Article pages: 168
CSS files: 8
Package files: 2
Docs files: 4
{
  "counts": {
    "main_pages": 9,
    "article_pages": 168,
    "css_files": 8,
    "package_files": 2,
    "docs_files": 4,
    "changed_files": 1
  },
  "remaining": {
    "Fokus Janten": 0,
    "Fokusjanten": 0,
    "FokusJanten": 0,
    "logo.png": 0
  },
  "new_color_presence": {
    "#166534": 175,
    "#052E16": 2,
    "#2D5016": 174
  }
}

Rebrand Arah Berita selesai ✅

PowerShell script to run:
Copy-Item -Path "articles.json" -Destination "articles.json.bak" -Force
Get-ChildItem -Recurse -Include *.html,*.css,*.md,*.json | ForEach-Object {
  $content = Get-Content $_.FullName -Raw -Encoding UTF8
  $content = $content -replace "Fokus\s?Janten", "Arah Berita"
  $content = $content -replace "Fokusjanten", "ArahBerita"
  $content = $content -replace "fokusjanten@gmail.com", "ArahBerita@gmail.com"
  $content = $content -replace "#0EA5E9", "#166534"
  $content = $content -replace "#22C55E", "#2D5016"
  $

In [ ]:
# 1. Import Libraries and Define Paths
from pathlib import Path
import re
import json
import shutil

root = Path(".").resolve()
html_files = list(root.rglob("*.html"))
css_files = list(root.rglob("*.css"))
json_files = list(root.rglob("*.json"))
md_files = list(root.rglob("*.md"))

def read_text(path):
    return path.read_text(encoding="utf-8", errors="replace")

def write_text(path, content):
    path.write_text(content, encoding="utf-8")

changed_files = set()

In [ ]:
# 2. Scan HTML and CSS Files
main_pages = [p for p in html_files if p.parent == root]
article_pages = [p for p in html_files if p.parent.name == "article"]
package_pages = [root / "package.json"]
if (root / "tools" / "package.json").exists():
    package_pages.append(root / "tools" / "package.json")
docs_pages = [root / "AUTOMATION_README.md", root / "GOOGLE_DRIVE_GUIDE.md", root / "GOOGLE_DRIVE_IMAGES_GUIDE.md", root / "netlify.toml"]

print(f"Main pages: {len(main_pages)}")
print(f"Article pages: {len(article_pages)}")
print(f"CSS files: {len(css_files)}")
print(f"Package files: {len(package_pages)}")
print(f"Docs files: {len([p for p in docs_pages if p.exists()])}")

In [ ]:
# 3. Update Branding Strings
branding_replacements = [
    (r"Fokus\s?Janten", "Arah Berita"),
    (r"Fokusjanten", "ArahBerita"),
    (r"fokusjanten", "ArahBerita"),
    (r"FokusJanten", "ArahBerita"),
    (r"fokus janten", "arah berita"),
    (r"fokusjanten@gmail\.com", "ArahBerita@gmail.com"),
    (r"Fokusjanten@gmail\.com", "ArahBerita@gmail.com"),
]

social_replacements = [
    (r"facebook\.com/[^\s'\"]+", "facebook.com/arahberita"),
    (r"twitter\.com/[^\s'\"]+", "twitter.com/arahberita"),
    (r"instagram\.com/[^\s'\"]+", "instagram.com/arahberita"),
    (r"youtube\.com/[^\s'\"]+", "youtube.com/arahberita"),
]

for path in main_pages + article_pages + docs_pages + css_files + package_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in branding_replacements + social_replacements:
        try:
            content = re.sub(old, new, content)
        except re.error:
            content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 4. Apply Global Theme Colors
color_map = {
    "#0EA5E9": "#166534",
    "#0ea5e9": "#166534",
    "#22C55E": "#2D5016",
    "#22c55e": "#2D5016",
    "#0F172A": "#052E16",
    "#0f172a": "#052E16",
    "#FFC107": "#166534",
    "#ffc107": "#166534",
    "#FFCC00": "#166534",
    "#ffcc00": "#166534",
    "#FC0": "#166534",
    "#fc0": "#166534",
    "#1E2024": "#052E16",
    "#1e2024": "#052E16",
}

for path in css_files + main_pages + article_pages + docs_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in color_map.items():
        content = content.replace(old, new)
    if path.name == "style.css":
        content = content.replace("--primary: #0EA5E9;", "--primary: #166534;")
        content = content.replace("--secondary: #22C55E;", "--secondary: #2D5016;")
        content = content.replace("--dark: #0F172A;", "--dark: #052E16;")
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 5. Replace Logo References with Text Logo
text_logo = '<span style="font-weight: bold; color: #166534; font-size: 24px; letter-spacing: -0.5px;">ARAH<span style="color: #2D5016; font-weight: normal; font-size: 18px; margin-left: 2px;">BERITA</span></span>'

for path in main_pages + article_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    content = content.replace('src="img/logo.png"', '')
    content = content.replace('src="../img/logo.png"', '')
    content = content.replace('src="./img/logo.png"', '')
    content = content.replace('alt="logo"', 'alt="ArahBerita"')
    content = content.replace('alt="Logo"', 'alt="ArahBerita"')
    content = content.replace('alt="LOGO"', 'alt="ArahBerita"')
    content = content.replace('FOKUS<span style="color: #22C55E; font-weight: normal; font-size: 18px; margin-left: 2px;">JANTEN</span>', text_logo)
    content = content.replace('FOKUS<span style="color: #22C55E; font-weight: normal; font-size: 18px;">JANTEN</span>', text_logo)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 6. Fix Quotation and Encoding
encoding_replacements = {
    '“': '"',
    '”': '"',
    '‘': "'",
    '’': "'",
    '–': '-',
    '—': '-',
    '\uFFFD': ' ',
    '\u00A0': ' ',
}

for path in html_files + md_files + css_files + json_files:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in encoding_replacements.items():
        content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 7. Patch Article Pages
for path in article_pages:
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    content = content.replace('src="../img/logo.png"', '')
    content = content.replace('src="img/logo.png"', '')
    content = content.replace('src="./img/logo.png"', '')
    content = content.replace('alt="logo"', 'alt="ArahBerita"')
    content = content.replace('alt="Logo"', 'alt="ArahBerita"')
    if 'FOKUS<span style="color: #22C55E; font-weight: normal' in content:
        content = content.replace('FOKUS<span style="color: #22C55E; font-weight: normal; font-size: 18px; margin-left: 2px;">JANTEN</span>', text_logo)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 8. Update package.json Metadata
pkg_updates = {
    'package.json': {
        '"name": "fokusjanten"': '"name": "arahberita"',
        '"description": "Fokus Janten website"': '"description": "Arah Berita website"',
    },
    'tools/package.json': {
        '"name": "arahberita-article-generator"': '"name": "arahberita-article-generator"',
        '"description": "Generator artikel otomatis dari Google Sheets untuk Fokus Janten"': '"description": "Generator artikel otomatis dari Google Sheets untuk Arah Berita"',
        '"author": "Fokus Janten Team"': '"author": "Arah Berita Team"',
    }
}

for rel, updates in pkg_updates.items():
    path = root / rel
    if not path.exists():
        continue
    content = read_text(path)
    original = content
    for old, new in updates.items():
        content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 9. Update Documentation Files
doc_replacements = [
    ('Fokus Janten', 'Arah Berita'),
    ('fokusjanten', 'arahberita'),
    ('Fokusjanten', 'ArahBerita'),
    ('#FFCC00', '#166534'),
    ('#ffc107', '#166534'),
    ('#1E2024', '#052E16'),
    ('/fokusjanten/', '/arahberita/'),
]

docs_to_update = [p for p in docs_pages if p.exists()]
for path in docs_to_update:
    content = read_text(path)
    original = content
    for old, new in doc_replacements:
        content = content.replace(old, new)
    if content != original:
        write_text(path, content)
        changed_files.add(str(path))

In [ ]:
# 10. Verify Rebranding and Generate Counts
remaining_patterns = [
    'Fokus Janten',
    'Fokusjanten',
    'FokusJanten',
    'logo.png',
]
new_colors = ['#166534', '#052E16', '#2D5016']

remaining = {}
for pattern in remaining_patterns:
    remaining[pattern] = []
    for path in html_files + css_files + md_files + json_files:
        if not path.exists():
            continue
        content = read_text(path)
        if pattern in content:
            remaining[pattern].append(str(path))

color_presence = {c: 0 for c in new_colors}
for path in html_files + css_files:
    if not path.exists():
        continue
    content = read_text(path)
    for c in new_colors:
        if c in content:
            color_presence[c] += 1

counts = {
    'main_pages': len(main_pages),
    'article_pages': len(article_pages),
    'css_files': len(css_files),
    'package_files': len(package_pages),
    'docs_files': len([p for p in docs_pages if p.exists()]),
    'changed_files': len(changed_files),
}

print(json.dumps({
    'counts': counts,
    'remaining': {k: len(v) for k, v in remaining.items()},
    'new_color_presence': color_presence,
}, indent=2))
print('\nRebrand Arah Berita selesai ✅')

In [ ]:
# 11. PowerShell Execution Script
shell_commands = [
    'Copy-Item -Path "articles.json" -Destination "articles.json.bak" -Force',
    'Get-ChildItem -Recurse -Include *.html,*.css,*.md,*.json | ForEach-Object {',
    '  $content = Get-Content $_.FullName -Raw -Encoding UTF8',
    '  $content = $content -replace "Fokus\\s?Janten", "Arah Berita"',
    '  $content = $content -replace "Fokusjanten", "ArahBerita"',
    '  $content = $content -replace "fokusjanten@gmail.com", "ArahBerita@gmail.com"',
    '  $content = $content -replace "#0EA5E9", "#166534"',
    '  $content = $content -replace "#22C55E", "#2D5016"',
    '  $content = $content -replace "#0F172A", "#052E16"',
    '  $content = $content -replace "#FFCC00", "#166534"',
    '  $content = $content -replace "#ffc107", "#166534"',
    '  $content = $content -replace "#1E2024", "#052E16"',
    '  $content = $content -replace "[“”]", "\""',
    '  $content = $content -replace "[‘’]", "\""',
    '  $content = $content -replace "[–—]", "-"',
    '  $content = $content -replace "\u00A0", " "',
    '  $content = $content -replace "\uFFFD", " "',
    '  $content = $content -replace "src=\"../img/logo.png\"", ""',
    '  $content = $content -replace "src=\"img/logo.png\"", ""',
    '  $content = $content -replace "alt=\"logo\"", "alt=\"ArahBerita\""',
    '  Set-Content -Path $_.FullName -Value $content -Encoding UTF8',
    '}',
]

print('\nPowerShell script to run:')
print('\n'.join(shell_commands))
